# ✍️ O-ISAC Survey Writing Blueprint Engine

Bu notebook, 76 adet IEEE COMST makalesini analiz ederek **senin yazacağın survey için bir "Yazım Reçetesi" (Recipe)** oluşturur. 

**Asıl Amacımız:** Bir makalenin iskeletini, görsel stratejisini ve "retorik gücünü" kopyalayarak, senin tek başına 6 kişilik bir dev ekip kalitesinde yazmanı sağlamaktır.

In [ ]:
# @title 1. Setup & Mount Drive
from google.colab import drive
from google.colab import userdata
import os
import sys
import json
import glob
import asyncio
import pandas as pd
from datetime import datetime
import nest_asyncio

nest_asyncio.apply()

drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/corpus_standardized')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'analysis')

print(f"✅ Project Root: {PROJECT_ROOT}")

In [ ]:
# @title 2. Install & Init GROQ
!pip install groq tqdm -q
from groq import Groq
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

## 🎓 The Writing Mentor System

In [ ]:
SYSTEM_PROMPT = """
You are the Senior Writing Mentor for IEEE Communications Surveys & Tutorials (COMST).
Your apprentice is writing a high-impact survey and needs a WRITING BLUEPRINT derived from the top papers in the field.

Do NOT just summarize. Provide a RECIPE for the apprentice to follow.

============================================================
WRITING BLUEPRINT SCHEMA (JSON)
============================================================
1. "rhetorical_skeleton": [ 
   { 
     "section": "string",
     "writing_objective": "What is the author trying to achieve here? (e.g., 'Selling the gap in current literature', 'Building a taxonomy from scratch')",
     "key_components": ["list of sub-elements that MUST be in this section"]
   }
],

2. "visual_roadmap": {
   "layout_logic": "Where are figures placed and WHY? (e.g., 'Uses a complex roadmap figure at 10% progress to set expectations')",
   "required_table_types": ["What kind of comparison tables are present? (e.g., 'Qualitative vs Quantitative Comparison', 'Protocol Field Summary')"]
},

3. "writing_style_secrets": {
   "phrasing_patterns": "Describe the sentence structure (e.g., 'Starts every section with a forward-looking summary', 'Heavy use of listicles for future directions')",
   "technical_depth_management": "How do they handle complex math without losing the reader?"
},

4. "actionable_mentor_advice": {
   "top_3_tips": ["Concrete advice for the apprentice to mirror this paper's success"],
   "mistakes_to_avoid": ["What did this paper do that was risky or could be improved?"]
}

============================================================
MENTORING GUIDELINES
============================================================
- Be encouraging but rigorous.
- Focus on HOW to write, not WHAT is written.
- Identify the 'Aha!' moment in the paper's structure.
"""

In [ ]:
# @title 3. Extraction & Mentoring Engine

async def extract_blueprint(paper_path, semaphore):
    async with semaphore:
        paper_id = os.path.basename(os.path.dirname(paper_path))
        with open(paper_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # Extract structural elements
        headers = "\n".join([line for line in content.split('\n') if line.startswith('#')])
        
        user_prompt = f"""
PAPER_ID: {paper_id}

SKELETON (Headers):
{headers}

FULL PAPER CONTENT (Sample for style and logic):
{content[:30000]}
"""

        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            
            result = json.loads(response.choices[0].message.content)
            result['paper_id'] = paper_id
            return result
        except Exception as e:
            return {"paper_id": paper_id, "error": str(e)}

In [ ]:
# @title 4. Generate All Blueprints

async def run_mentoring():
    papers = sorted(glob.glob(os.path.join(DATA_DIR, "COMST_*/COMST_*.md")))
    print(f"🎓 Generating Writing Blueprints for {len(papers)} papers...")
    
    semaphore = asyncio.Semaphore(2) 
    tasks = [extract_blueprint(p, semaphore) for p in papers]
    
    blueprints = []
    from tqdm.notebook import tqdm
    for f in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        blueprints.append(await f)
        
    output_path = os.path.join(OUTPUT_DIR, "writing_blueprints_master.json")
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(blueprints, f, indent=4)
        
    print(f"\n✨ Masters' Secrets revealed at: {output_path}")
    return blueprints

blueprints = await run_mentoring()